# Session 06 Practice: Dynamic Motion Prediction & Time-to-Collision (TTC) Risk
**Course:** Motion Planning for Autonomous Vehicles (Planning P2)  
**Week 3 | Session 6**  

---

### 🎯 Learning Objectives
1. Implement physics-based dynamic motion prediction models (**Constant Velocity (CV)**).
2. Represent vehicles as rotated 2D **Oriented Bounding Boxes (OBB)** using `Shapely`.
3. Compute spatial-temporal intersection collisions and compute **Time-to-Collision (TTC)**.
4. Visualize dynamic prediction horizons, collision cones, and trajectory risk metrics using **Plotly**.

---

### 📐 Mathematical Formulation

#### Constant Velocity (CV) Prediction
For an obstacle with state $S_0 = (x_0, y_0, 	heta_0, v_0)$:

$$
egin{aligned}
x(t) &= x_0 + v_0 \cos(	heta_0) t \
y(t) &= y_0 + v_0 \sin(	heta_0) t
\end{aligned}
$$

#### Time-to-Collision (TTC)
$$	ext{TTC} = \min \left\{ t \ge 0 \;\middle|\; \mathcal{P}_{	ext{ego}}(t) \cap \mathcal{P}_{	ext{obs}}(t) 
eq \emptyset ight\}$$


In [ ]:
import math
import numpy as np
import polars as pl
import plotly.graph_objects as go
from shapely.geometry import Polygon
from shapely.affinity import rotate, translate
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "plotly_white"


## 1. Oriented Bounding Box (OBB) & Prediction Models

In [ ]:
def create_vehicle_polygon(center_x: float, center_y: float, yaw_rad: float,
                            length: float = 4.5, width: float = 2.0) -> Polygon:
    half_l = length / 2.0
    half_w = width / 2.0
    base_box = Polygon([
        (-half_l, -half_w),
        (half_l, -half_w),
        (half_l, half_w),
        (-half_l, half_w)
    ])
    rotated = rotate(base_box, math.degrees(yaw_rad), origin=(0, 0))
    final_poly = translate(rotated, xoff=center_x, yoff=center_y)
    return final_poly

def predict_constant_velocity(x0: float, y0: float, yaw0: float, v0: float,
                              dt: float = 0.1, horizon: float = 4.0) -> list[dict]:
    states = []
    num_steps = int(horizon / dt)
    for i in range(num_steps):
        t = i * dt
        x = x0 + v0 * math.cos(yaw0) * t
        y = y0 + v0 * math.sin(yaw0) * t
        states.append({"time": round(t, 2), "x": x, "y": y, "yaw": yaw0, "v": v0})
    return states

def evaluate_ttc(ego_states: list[dict], obs_states: list[dict],
                 ego_dim: tuple = (4.5, 2.0), obs_dim: tuple = (4.5, 2.0)) -> tuple[float, int]:
    for i, (ego_st, obs_st) in enumerate(zip(ego_states, obs_states)):
        ego_poly = create_vehicle_polygon(ego_st["x"], ego_st["y"], ego_st["yaw"], ego_dim[0], ego_dim[1])
        obs_poly = create_vehicle_polygon(obs_st["x"], obs_st["y"], obs_st["yaw"], obs_dim[0], obs_dim[1])
        
        if ego_poly.intersects(obs_poly):
            return ego_st["time"], i
            
    return float('inf'), -1


## 2. Dynamic Intersection Crossing Scenario

In [ ]:
def run_intersection_risk_assessment(ego_speed: float = 10.0, obs_speed: float = 12.0):
    dt = 0.1
    horizon = 5.0
    num_steps = int(horizon / dt)
    
    # Ego: Moving Northbound from (0, -25)
    ego_states = []
    for i in range(num_steps):
        t = i * dt
        ego_states.append({
            "time": round(t, 2),
            "x": 0.0,
            "y": -25.0 + ego_speed * t,
            "yaw": math.pi / 2,
            "v": ego_speed
        })
        
    # Dynamic Obstacle: Moving Eastbound from (-30, 0)
    obs_states = predict_constant_velocity(x0=-30.0, y0=0.0, yaw0=0.0, v0=obs_speed, dt=dt, horizon=horizon)
    
    ttc, col_idx = evaluate_ttc(ego_states, obs_states)
    
    records = []
    for i in range(num_steps):
        ego = ego_states[i]
        obs = obs_states[i]
        dist = math.hypot(ego["x"] - obs["x"], ego["y"] - obs["y"])
        records.append({
            "time_s": ego["time"],
            "ego_x": ego["x"],
            "ego_y": ego["y"],
            "obs_x": obs["x"],
            "obs_y": obs["y"],
            "center_dist_m": dist,
            "is_colliding": (i == col_idx)
        })
        
    return pl.DataFrame(records), ttc, ego_states, obs_states

df_risk, ttc, ego_st, obs_st = run_intersection_risk_assessment()
print(f"Computed Time-To-Collision (TTC): {ttc:.2f} seconds")


## 3. Interactive Collision Zone & Spatial-Temporal Plots

In [ ]:
# 1. 2D Intersection Spatial Trajectory & OBB Footprints
fig_map = go.Figure()

# Paths
fig_map.add_trace(go.Scatter(
    x=df_risk["ego_x"], y=df_risk["ego_y"], mode="lines+markers",
    name="Ego Vehicle (Northbound)", line=dict(color="#1f77b4", width=3)
))
fig_map.add_trace(go.Scatter(
    x=df_risk["obs_x"], y=df_risk["obs_y"], mode="lines+markers",
    name="Obstacle Vehicle (Eastbound)", line=dict(color="#ff7f0e", width=3)
))

# Draw OBB bounding boxes every 0.5 seconds
for idx in range(0, len(ego_st), 5):
    e = ego_st[idx]
    o = obs_st[idx]
    
    # Ego box
    ep = create_vehicle_polygon(e["x"], e["y"], e["yaw"])
    ex, ey = ep.exterior.xy
    fig_map.add_trace(go.Scatter(x=list(ex), y=list(ey), mode="lines",
                                 line=dict(color="rgba(31, 119, 180, 0.4)", width=1.5),
                                 showlegend=False, hoverinfo="none"))
    
    # Obs box
    op = create_vehicle_polygon(o["x"], o["y"], o["yaw"])
    ox, oy = op.exterior.xy
    fig_map.add_trace(go.Scatter(x=list(ox), y=list(oy), mode="lines",
                                 line=dict(color="rgba(255, 127, 14, 0.4)", width=1.5),
                                 showlegend=False, hoverinfo="none"))

# Highlight collision zone
col_row = df_risk.filter(pl.col("is_colliding"))
if len(col_row) > 0:
    fig_map.add_trace(go.Scatter(
        x=[col_row["ego_x"][0], col_row["obs_x"][0]],
        y=[col_row["ego_y"][0], col_row["obs_y"][0]],
        mode="markers",
        marker=dict(size=20, color="crimson", symbol="x-thin", line=dict(width=4)),
        name=f"Collision Zone (TTC = {ttc:.2f}s)"
    ))

fig_map.update_layout(
    title="Intersection Conflict Zone: Dynamic Motion Prediction & OBB Intersections",
    xaxis_title="Global X (m)",
    yaxis_title="Global Y (m)",
    xaxis=dict(scaleanchor="y", scaleratio=1),
    width=800,
    height=800
)
fig_map.show()


In [ ]:
# 2. Inter-Vehicle Distance Evolution & TTC Sensitivity
fig_dist = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Inter-Vehicle Distance vs Time", "TTC Sensitivity across Obstacle Speeds")
)

fig_dist.add_trace(
    go.Scatter(x=df_risk["time_s"], y=df_risk["center_dist_m"], mode="lines+markers",
               line=dict(color="#9467bd", width=3), name="Center-to-Center Distance"),
    row=1, col=1
)
fig_dist.add_hline(y=4.5, line_dash="dash", line_color="red", annotation_text="Collision Threshold (4.5m)", row=1, col=1)

# Speed sensitivity sweep
obs_speeds = np.linspace(5.0, 20.0, 30)
ttc_sweep = []
for spd in obs_speeds:
    _, s_ttc, _, _ = run_intersection_risk_assessment(ego_speed=10.0, obs_speed=spd)
    ttc_sweep.append(s_ttc if s_ttc != float('inf') else None)

fig_dist.add_trace(
    go.Scatter(x=obs_speeds, y=ttc_sweep, mode="lines+markers",
               line=dict(color="#2ca02c", width=2.5), name="TTC vs Speed"),
    row=1, col=2
)

fig_dist.update_layout(title="Dynamic Collision Risk Telemetry", height=420, showlegend=False)
fig_dist.show()


## 4. Verification Tests

In [ ]:
assert 2.0 <= ttc <= 3.0, f"Expected TTC between 2.0s and 3.0s, got {ttc}s"
assert len(col_row) == 1, "Collision point was not correctly flagged!"

print("✅ Dynamic prediction and TTC verification tests passed successfully!")
